# Saved evaluation results

> **SIMULATION ONLY.** These scores measure reconstruction of the synthetic simulator. They are not validated crop accuracy and do not establish real storage life.

This notebook reads saved JSON/CSV results. It does not train models, alter the split, or modify the dataset. Re-run the project training/evaluation commands before refreshing results.

In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "artifacts").exists():
    ROOT = ROOT.parent
ARTIFACTS = ROOT / "artifacts"


In [2]:
status = json.loads((ARTIFACTS / "status_metrics.json").read_text())
advanced = json.loads((ARTIFACTS / "advanced_status_metrics.json").read_text())
survival = json.loads((ARTIFACTS / "survival_metrics.json").read_text())
diagnostics = json.loads((ARTIFACTS / "diagnostics" / "diagnostics_metrics.json").read_text())
print(status["warning"])
print(f"Fixed grouped holdout: {advanced['split']['train_lots']} train lots / {advanced['split']['test_lots']} test lots; test IDs unchanged: {advanced['split']['test_ids_unchanged']}")
print(f"Grouped validation: {advanced['cv']['folds']} folds grouped by sim_lot_id")
print(f"Test rows: {advanced['split']['test_rows']:,}; independent held-out lots: {advanced['split']['test_lots']}")


SIMULATION ONLY: reconstruction of simulator assumptions, not validated crop accuracy.
Fixed grouped holdout: 400 train lots / 100 test lots; test IDs unchanged: True
Grouped validation: 5 folds grouped by sim_lot_id
Test rows: 24,000; independent held-out lots: 100


In [3]:
rows = []
for r in status["results"]:
    rows.append({
        "features": r["feature_set"], "model": r["model"],
        "accuracy": r["accuracy"], "macro_F1": r["macro_f1"],
        "spoiled_recall": r["per_class"]["spoiled"]["recall"],
        "log_loss": r["log_loss"], "Brier": r["multiclass_brier"],
    })
comparison = pd.DataFrame(rows)
print(comparison.to_string(index=False, float_format=lambda x: f"{x:.4f}"))


        features         model  accuracy  macro_F1  spoiled_recall  log_loss  Brier
environment_only      majority    0.3682    0.1794          0.0000    1.0962 0.6650
environment_only random_forest    0.8186    0.8226          0.8399    0.4077 0.2553
environment_only       xgboost    0.8321    0.8360          0.8445    0.3921 0.2382
         mq_only      majority    0.3682    0.1794          0.0000    1.0962 0.6650
         mq_only random_forest    0.9220    0.9230          0.9526    0.1908 0.1138
         mq_only       xgboost    0.9240    0.9251          0.9526    0.1738 0.1084
        combined      majority    0.3682    0.1794          0.0000    1.0962 0.6650
        combined random_forest    0.9300    0.9309          0.9587    0.1783 0.1055
        combined       xgboost    0.9291    0.9300          0.9562    0.1622 0.1004


In [4]:
selected = advanced["combined"]["xgboost"]
per_crop = []
for crop, result in selected["per_crop"].items():
    per_crop.append({
        "crop": crop, "rows": result["rows"], "accuracy": result["per_class"]["accuracy"],
        "macro_F1": result["macro_f1"], "spoiled_recall": result["per_class"]["spoiled"]["recall"],
        "warning_recall": result["per_class"]["warning"]["recall"],
        "acceptable_recall": result["per_class"]["acceptable"]["recall"],
    })
print("Combined XGBoost per-crop test results")
print(pd.DataFrame(per_crop).to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print("\nOverall per-class recall")
print(pd.DataFrame([{"class": k, "recall": v["recall"], "support": int(v["support"])} for k,v in selected["per_class"].items() if k in {"acceptable", "warning", "spoiled"}]).to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print("\nConfusion matrix (actual rows; predicted columns; acceptable, warning, spoiled)")
print(pd.DataFrame(selected["confusion_matrix"], index=selected["confusion_matrix_labels_order"], columns=selected["confusion_matrix_labels_order"]).to_string())


Combined XGBoost per-crop test results
   crop  rows  accuracy  macro_F1  spoiled_recall  warning_recall  acceptable_recall
  maize  4800    0.9206    0.9232          0.9394          0.8960             0.9346
  onion  4800    0.9142    0.9165          0.9391          0.8520             0.9518
 potato  4800    0.9383    0.9400          0.9621          0.9274             0.9296
spinach  4800    0.9587    0.9506          0.9718          0.9302             0.9644
  wheat  4800    0.9135    0.9185          0.9536          0.9101             0.8924

Overall per-class recall
     class  recall  support
acceptable  0.9311     7269
   warning  0.9032     8836
   spoiled  0.9562     7895

Confusion matrix (actual rows; predicted columns; acceptable, warning, spoiled)
            acceptable  warning  spoiled
acceptable        6768      501        0
warning            627     7981      228
spoiled              0      346     7549


In [5]:
print("Per-crop failure/censoring counts on held-out trajectories")
event_rows = []
for crop, counts in survival["per_crop_event_censoring"].items():
    test = counts["test"]
    event_rows.append({"crop":crop,"test_trajectories":test["trajectories"],"observed_failures":test["events"],"right_censored":test["right_censored"]})
print(pd.DataFrame(event_rows).to_string(index=False))
print("\nSupported crop-specific calendar horizon results: censoring-aware Brier score")
horizon_rows=[]
for crop, h in survival["crop_specific_calendar_horizons"].items():
    row={"crop":crop,"horizon_day":h["calendar_horizon_day"],"interval_days":h["interval_days"]}
    for model in ["crop_specific_baseline","discrete_hazard_logistic","discrete_hazard_xgboost"]:
        row[model+"_Brier"]=h[model]["brier"]
        row[model+"_mean_pred"]=h[model]["calibration"]["mean_predicted"]
    row["observed_frac"]=h["discrete_hazard_xgboost"]["calibration"]["observed_fraction"]
    horizon_rows.append(row)
print(pd.DataFrame(horizon_rows).to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print("\nBrier scores above are evaluated at the stated horizon with censoring adjustment. Horizons/cadence differ by crop; these values do not imply continuous days-left accuracy.")


Per-crop failure/censoring counts on held-out trajectories
   crop  test_trajectories  observed_failures  right_censored
  maize                120                 78              42
  onion                120                 73              47
 potato                120                 79              41
spinach                120                113               7
  wheat                120                 65              55

Supported crop-specific calendar horizon results: censoring-aware Brier score
   crop  horizon_day  interval_days  crop_specific_baseline_Brier  crop_specific_baseline_mean_pred  discrete_hazard_logistic_Brier  discrete_hazard_logistic_mean_pred  discrete_hazard_xgboost_Brier  discrete_hazard_xgboost_mean_pred  observed_frac
  maize     105.0000         7.0000                        0.0198                            0.0225                          0.0330                              0.0847                         0.0070                             0.0168        

## Interpretation

- The saved scores are from synthetic rows and synthetic outcomes. They cannot be used as evidence of field accuracy.
- Rows repeat within simulated trajectories; the split keeps all paths and readings from each starting lot together. The independent test set contains 100 starting lots.
- Probability metrics describe the model's probabilities on this simulator test set. A real deployment needs paired sensor histories and independent quality inspections.
- Discrete hazard risk is for the next inspection interval. It is not a continuous shelf-life clock. Future-path projections need explicit future environment and MQ readings and do not estimate a causal cooling benefit.